### 🔄 프로젝트 초기화 (Reset Project)
아래 코드를 실행하여 런타임을 다시 시작하고 깨끗한 상태에서 시작하세요.

In [ ]:
import os

# 현재 작업 디렉토리의 파일들 정리 (선택 사항)
# !rm -rf ./my_finetuned_gpt2 ./distilled_gpt2_student model_utils.py

print("🔄 런타임을 종료합니다. 잠시 후 자동으로 재연결되면 다시 시작해 주세요.")
os.kill(os.getpid(), 9)

# Uploading Models with KerasHub

**Author:** [Samaneh Saadat](https://github.com/SamanehSaadat/), [Matthew Watson](https://github.com/mattdangerw/)<br>
**Date created:** 2024/04/29<br>
**Last modified:** 2024/04/29<br>
**Description:** An introduction on how to upload a fine-tuned KerasHub model to model hubs.

# Introduction

Fine-tuning a machine learning model can yield impressive results for specific tasks.
Uploading your fine-tuned model to a model hub allows you to share it with the broader community.
By sharing your models, you'll enhance accessibility for other researchers and developers,
making your contributions an integral part of the machine learning landscape.
This can also streamline the integration of your model into real-world applications.

This guide walks you through how to upload your fine-tuned models to popular model hubs such as
[Kaggle Models](https://www.kaggle.com/models) and [Hugging Face Hub](https://huggingface.co/models).

# Setup

Let's start by installing and importing all the libraries we need. We use KerasHub for this guide.

In [ ]:
!pip install -q --upgrade keras-hub huggingface-hub kagglehub

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras_hub

In [ ]:
!pip install -q --upgrade keras-hub huggingface-hub kagglehub

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras_hub

In [ ]:
import keras_hub

# Data

We can use the IMDB reviews dataset for this guide. Let's load the dataset from `tensorflow_dataset`.

In [ ]:
import tensorflow_datasets as tfds

imdb_train, imdb_test = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
    batch_size=4,
)

In [ ]:
causal_lm = keras_hub.models.CausalLM.from_preset("gpt2_base_en")

In [ ]:
causal_lm = keras_hub.models.CausalLM.from_preset("gpt2_base_en")

We only use a small subset of the training samples to make the guide run faster.
However, if you need a higher quality model, consider using a larger number of training samples.

# Task Upload

A `keras_hub.models.Task`, wraps a `keras_hub.models.Backbone` and a `keras_hub.models.Preprocessor` to create
a model that can be directly used for training, fine-tuning, and prediction for a given text problem.
In this section, we explain how to create a `Task`, fine-tune and upload it to a model hub.

In [ ]:
imdb_train = imdb_train.take(100)

## Load Model

If you want to build a Causal LM based on a base model, simply call `keras_hub.models.CausalLM.from_preset`
and pass a built-in preset identifier.

## Fine-tune Model

After loading the model, you can call `.fit()` on the model to fine-tune it.
Here, we fine-tune the model on the IMDB reviews which makes the model movie domain-specific.

In [ ]:
# Drop labels and keep the review text only for the Causal LM.
imdb_train_reviews = imdb_train.map(lambda x, y: x)

# Fine-tune the Causal LM.
causal_lm.fit(imdb_train_reviews)

In [ ]:
# imdb_train 데이터셋의 element_spec 확인
print("imdb_train element_spec:", imdb_train.element_spec)

# 첫 번째 배치에서 샘플 데이터 확인
for text_batch, label_batch in imdb_train.take(1):
    print("\nSample text from the first batch:", text_batch[0].numpy().decode('utf-8'))
    print("Sample label from the first batch:", label_batch[0].numpy())
    break

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
import keras
import keras_hub

# 1. 모델 초기화
try:
    print("📦 GPT-2 모델 프리셋 로드 중...")
    causal_lm = keras_hub.models.CausalLM.from_preset("gpt2_base_en")

    # 2. 데이터셋 로드 및 안정적인 전처리
    imdb_train = tfds.load("imdb_reviews", split="train", as_supervised=True)
    # 빠른 시각화를 위해 샘플 수를 제한합니다.
    imdb_train_subset = imdb_train.take(100).batch(4)
    imdb_train_texts = imdb_train_subset.map(lambda text, label: text)

    # 3. 모델 학습 및 결과 저장
    print("🚀 모델 학습을 시작합니다 (1 Epoch)... ")
    history = causal_lm.fit(imdb_train_texts, epochs=1)
    print("✅ 학습이 완료되었습니다.")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

### 📊 학습 지표 시각화 (Training Metrics Visualization)

모델의 학습 과정에서 기록된 손실(Loss)과 정확도(Accuracy) 변화를 그래프로 시각화합니다.

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    if history is None or not hasattr(history, 'history'):
        print("❌ 시각화할 학습 기록이 없습니다.")
        return

    plt.figure(figsize=(12, 4))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Loss', color='#4A90E2', marker='o')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    acc_key = 'sparse_categorical_accuracy' if 'sparse_categorical_accuracy' in history.history else 'accuracy'
    if acc_key in history.history:
        plt.plot(history.history[acc_key], label='Accuracy', color='#F5A623', marker='o')
        plt.title('Training Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()

    plt.tight_layout()
    plt.show()

try:
    plot_history(history)
except NameError:
    print("❌ 'history' 객체가 정의되지 않았습니다. 학습 셀을 먼저 실행해 주세요.")

### Text Generation Test

Let's test our fine-tuned GPT-2 model to see if it has learned to generate text in the style of IMDB movie reviews.

In [ ]:
prompt = "This movie was"
output = causal_lm.generate(prompt, max_length=100)
print(output)

### Improving Generation Quality with Sampling

By default, generation might be repetitive. We can use `sampler` arguments to make the output more natural. `top_p` and `temperature` are commonly used for fine-tuned LLMs.

In [ ]:
# To improve quality, we compile the model with a specific sampler
causal_lm.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))

# Now we generate without passing 'sampler' to the generate call
output_tuned = causal_lm.generate(
    "This movie was",
    max_length=100
)

print("--- Tuned Generation (Top-P) ---")
print(output_tuned)

### 모델 저장하기

학습이 완료된 모델을 `save_to_preset`을 사용하여 로컬 디렉토리에 저장합니다.

In [ ]:
# 저장할 디렉토리 경로 설정
preset_dir = "./my_finetuned_gpt2"

# 모델 저장
causal_lm.save_to_preset(preset_dir)

print(f"모델이 '{preset_dir}' 경로에 저장되었습니다.")
# 저장된 파일 목록 확인
import os
print("저장된 파일:", os.listdir(preset_dir))

### Kaggle에 모델 업로드하기

Kaggle에 모델을 올리려면 먼저 Kaggle API 인증이 필요합니다. 아래 셀을 실행하여 로그인하고 업로드를 진행하세요.

In [ ]:
import kagglehub
import os

# 1단계: 이 셀을 실행하고 나타나는 위젯에 Kaggle 계정 정보를 입력하세요.
# 로그인 버튼을 누른 후 'Kaggle credentials set' 메시지를 확인해야 합니다.
kagglehub.login()

## Save the Model Locally

To upload a model, you need to first save the model locally using `save_to_preset`.

In [ ]:
preset_dir = "./gpt2_imdb"
causal_lm.save_to_preset(preset_dir)

Let's see the saved files.

In [ ]:
os.listdir(preset_dir)

### Load a Locally Saved Model

A model that is saved to a local preset can be loaded using `from_preset`.
What you save in, is what you get back out.

In [ ]:
causal_lm = keras_hub.models.CausalLM.from_preset(preset_dir)

You can also load the `keras_hub.models.Backbone` and `keras_hub.models.Tokenizer` objects from this preset directory.
Note that these objects are equivalent to `causal_lm.backbone` and `causal_lm.preprocessor.tokenizer` above.

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(preset_dir)
tokenizer = keras_hub.models.Tokenizer.from_preset(preset_dir)

## Upload the Model to a Model Hub

After saving a preset to a directory, this directory can be uploaded to a model hub such as Kaggle or Hugging Face directly from the KerasHub library.
To upload the model to Kaggle, the URI must start with `kaggle://` and to upload to Hugging Face, it should start with `hf://`.

### Upload to Kaggle

To upload a model to Kaggle, first, we need to authenticate with Kaggle.
This can in one of the following ways:
1. Set environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY`.
2. Provide a local `~/.kaggle/kaggle.json`.
3. Call `kagglehub.login()`.

Let's make sure we are logged in before continuing.

In [ ]:
import kagglehub

if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
    kagglehub.login()

To upload a model we can use `keras_hub.upload_preset(uri, preset_dir)` API where `uri` has the format of
`kaggle://<KAGGLE_USERNAME>/<MODEL>/Keras/<VARIATION>` for uploading to Kaggle and `preset_dir` is the directory that the model is saved in.

Running the following uploads the model that is saved in `preset_dir` to Kaggle:

In [ ]:
import keras_hub
import kagglehub

# 2단계: 로그인이 완료되었으므로 업로드를 수행합니다.
try:
    user_info = kagglehub.whoami()
    kaggle_username = user_info["username"]
    print(f"인증 확인됨: {kaggle_username}")

    # 업로드 설정
    model_name = "gpt2-imdb-finetuned"
    variation_name = "v1"
    kaggle_uri = f"kaggle://{kaggle_username}/{model_name}/keras/{variation_name}"
    # 앞서 저장한 모델 경로 사용
    current_preset_dir = "./my_finetuned_gpt2"

    print(f"'{current_preset_dir}'의 모델을 {kaggle_uri}로 업로드 중...")
    keras_hub.upload_preset(kaggle_uri, current_preset_dir)
    print("🎉 업로드가 완료되었습니다! Kaggle Model Hub에서 확인해 보세요.")

except Exception as e:
    print(f"업로드 중 오류 발생: {e}")

In [ ]:
import kagglehub

# Kaggle 인증을 다시 수행합니다.
# 위젯이 나타나면 정보를 입력하고 반드시 'Login' 버튼을 클릭하세요.
kagglehub.login()

### Upload to Hugging Face

To upload a model to Hugging Face, first, we need to authenticate with Hugging Face.
This can in one of the following ways:
1. Set environment variables `HF_USERNAME` and `HF_TOKEN`.
2. Call `huggingface_hub.notebook_login()`.

Let's make sure we are logged in before coninuing.

In [ ]:
import huggingface_hub

if "HF_USERNAME" not in os.environ or "HF_TOKEN" not in os.environ:
    huggingface_hub.notebook_login()

`keras_hub.upload_preset(uri, preset_dir)` can be used to upload a model to Hugging Face if `uri` has the format of
`kaggle://<HF_USERNAME>/<MODEL>`.

Running the following uploads the model that is saved in `preset_dir` to Hugging Face:

In [ ]:
hf_username = huggingface_hub.whoami()["name"]
hf_uri = f"hf://{hf_username}/gpt2_imdb"
keras_hub.upload_preset(hf_uri, preset_dir)

### Hugging Face Hub 로그인

Hugging Face에 모델을 업로드하거나 비공개 모델을 불러오려면 인증이 필요합니다. [Hugging Face Settings](https://huggingface.co/settings/tokens)에서 토큰을 생성한 뒤 아래 셀을 실행하여 입력해 주세요.

In [ ]:
from huggingface_hub import notebook_login

# Hugging Face 로그인 위젯 호출
notebook_login()

In [ ]:
import huggingface_hub
import keras_hub

try:
    # Get the logged-in user's name
    hf_username = huggingface_hub.whoami()["name"]
    print(f"Authenticated as: {hf_username}")

    # Set the model identifier
    model_id = "gpt2-imdb-finetuned"
    hf_uri = f"hf://{hf_username}/{model_id}"

    # Use the existing preset directory we saved earlier
    current_preset_dir = "./my_finetuned_gpt2"

    print(f"Uploading model from '{current_preset_dir}' to '{hf_uri}'...")
    keras_hub.upload_preset(hf_uri, current_preset_dir)
    print(f"\u2728 Successfully uploaded to: https://huggingface.co/{hf_username}/{model_id}")

except Exception as e:
    print(f"An error occurred during upload: {e}")

In [ ]:
from huggingface_hub import notebook_login

# This will open a widget to enter your Hugging Face Access Token
notebook_login()

In [ ]:
import huggingface_hub
import keras_hub

try:
    # Get the logged-in user's name
    user_info = huggingface_hub.whoami()
    hf_username = user_info["name"]
    print(f"Authenticated as: {hf_username}")

    # Set the model identifier
    model_id = "gpt2-imdb-finetuned"
    hf_uri = f"hf://{hf_username}/{model_id}"

    # Use the existing preset directory we saved earlier
    current_preset_dir = "./my_finetuned_gpt2"

    print(f"Uploading model from '{current_preset_dir}' to '{hf_uri}'...")
    keras_hub.upload_preset(hf_uri, current_preset_dir)
    print(f"\u2728 Successfully uploaded to: https://huggingface.co/{hf_username}/{model_id}")

except Exception as e:
    print(f"An error occurred: {e}")
    print("Make sure you have successfully logged in using the notebook_login() widget above before running this cell.")

In [ ]:
import huggingface_hub
import keras_hub

try:
    # Get the logged-in user's name from Hugging Face
    user_info = huggingface_hub.whoami()
    hf_username = user_info["name"]
    print(f"Authenticated as: {hf_username}")

    # Set the repository name and URI
    model_id = "gpt2-imdb-finetuned"
    hf_uri = f"hf://{hf_username}/{model_id}"

    # The local directory where your model was saved earlier
    local_preset_dir = "./my_finetuned_gpt2"

    print(f"Uploading model to {hf_uri}...")
    # Upload the preset to Hugging Face Hub
    keras_hub.upload_preset(hf_uri, local_preset_dir)

    print(f"\u2728 Successfully uploaded! View your model here: https://huggingface.co/{hf_username}/{model_id}")

except Exception as e:
    print(f"An error occurred during upload: {e}")
    print("Please ensure you have completed the 'notebook_login()' step successfully.")

## Load a User Uploaded Model

After verifying that the model is uploaded to Kaggle, we can load the model by calling `from_preset`.

```python
causal_lm = keras_hub.models.CausalLM.from_preset(
    f"kaggle://{kaggle_username}/gpt2/keras/gpt2_imdb"
)
```

We can also load the model uploaded to Hugging Face by calling `from_preset`.

```python
causal_lm = keras_hub.models.CausalLM.from_preset(f"hf://{hf_username}/gpt2_imdb")
```

# Classifier Upload

Uploading a classifier model is similar to Causal LM upload.
To upload the fine-tuned model, first, the model should be saved to a local directory using `save_to_preset`
API and then it can be uploaded via `keras_hub.upload_preset`.

In [ ]:
# Load the base model.
classifier = keras_hub.models.Classifier.from_preset(
    "bert_tiny_en_uncased", num_classes=2
)

# Fine-tune the classifier.
classifier.fit(imdb_train)

# Save the model to a local preset directory.
preset_dir = "./bert_tiny_imdb"
classifier.save_to_preset(preset_dir)

# Upload to Kaggle.
keras_hub.upload_preset(
    f"kaggle://{kaggle_username}/bert/keras/bert_tiny_imdb", preset_dir
)

After verifying that the model is uploaded to Kaggle, we can load the model by calling `from_preset`.

```python
classifier = keras_hub.models.Classifier.from_preset(
    f"kaggle://{kaggle_username}/bert/keras/bert_tiny_imdb"
)
```

In [ ]:
from huggingface_hub import notebook_login

# Open the Hugging Face login widget
notebook_login()

In [ ]:
import huggingface_hub
import keras_hub

try:
    # This will work once you have logged in via the widget above
    user_info = huggingface_hub.whoami()
    hf_username = user_info["name"]

    model_id = "gpt2-imdb-finetuned"
    hf_uri = f"hf://{hf_username}/{model_id}"
    local_preset_dir = "./my_finetuned_gpt2"

    print(f"Uploading model to {hf_uri}...")
    keras_hub.upload_preset(hf_uri, local_preset_dir)
    print(f"\u2728 Successfully uploaded to: https://huggingface.co/{hf_username}/{model_id}")

except Exception as e:
    print(f"Upload failed: {e}")
    print("Make sure you have successfully logged in using the widget above.")

### Inference Test with Hugging Face Model

Now that the model is uploaded, let's test if we can load it directly from the Hugging Face Hub and generate some text.

In [ ]:
import keras_hub

# Define the Hugging Face URI
hf_model_uri = "hf://hinttt/gpt2-imdb-finetuned"

# Load the model from the Hub
print(f"Loading model from {hf_model_uri}...")
reloaded_lm = keras_hub.models.CausalLM.from_preset(hf_model_uri)

# Run a test generation
test_prompt = "The acting in this movie was"
output = reloaded_lm.generate(test_prompt, max_length=100)

print("\n--- Inference Output ---")
print(output)

### Updating Model Card (README.md)

You can update your model's documentation and metadata by uploading a `README.md` file.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Create a simple README content
readme_content = """---
language: en
license: mit
tags:
- keras-hub
- gpt2
- movie-reviews
---

# GPT-2 IMDB Fine-tuned
This model is a fine-tuned version of GPT-2 on the IMDB reviews dataset using KerasHub.
"""

with open("README.md", "w") as f:
    f.write(readme_content)

# Upload the README to the hub
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id="hinttt/gpt2-imdb-finetuned",
    repo_type="model"
)

print("Model card updated successfully!")

### Detailed Model Card Example

Below is a more comprehensive template for your `README.md` including usage instructions and model details.

In [ ]:
readme_template = """---
language: en
license: mit
tags:
- keras-hub
- gpt2
- text-generation
- movie-reviews
datasets:
- imdb
---

# GPT-2 Fine-tuned on IMDB

This model is a fine-tuned version of GPT-2 using [KerasHub](https://github.com/keras-team/keras-hub). It has been trained on the IMDB movie reviews dataset to generate text in the style of movie critiques.

## Model Description
- **Model Type:** Causal Language Model
- **Base Model:** GPT-2 Base
- **Framework:** Keras 3 (with JAX backend)
- **Task:** Text Generation

## How to use

You can load and use this model directly with KerasHub:

```python
import keras_hub
import keras

# Load the model
model = keras_hub.models.CausalLM.from_preset("hf://{hf_username}/{model_id}")

# Generate text
output = model.generate("This movie was", max_length=100)
print(output)
```

## Training Details
- **Dataset:** IMDB Reviews (subset)
- **Epochs:** 1
- **Optimizer:** AdamW (default KerasHub settings)
"""

# Fill in the variables
final_readme = readme_template.format(hf_username=hf_username, model_id=model_id)

with open("README_detailed.md", "w") as f:
    f.write(final_readme)

print("Detailed README template created as 'README_detailed.md'.")
print("You can use api.upload_file() as shown above to update your Hub repo with this content.")

In [ ]:
# 상세 템플릿을 실제 README.md로 업로드
api.upload_file(
    path_or_fileobj="README_detailed.md",
    path_in_repo="README.md",
    repo_id=f"{hf_username}/{model_id}",
    repo_type="model"
)

print(f"https://huggingface.co/{hf_username}/{model_id} 에 상세 모델 카드가 반영되었습니다!")

In [ ]:
from model_utils import load_model_weights_safely

# 파일에서 불러온 함수로 다시 테스트
test_model = load_model_weights_safely(
    preset_name='gpt2_base_en',
    weights_path=downloaded_path
)
print('파일 임포트 및 함수 실행 성공!')

# [Project] KerasHub를 활용한 GPT-2 영화 리뷰 생성기 파인튜닝 및 배포

## 1. 프로젝트 개요
본 프로젝트는 GPT-2 기반의 거대 언어 모델(LLM)을 KerasHub를 사용하여 특정 도메인(IMDB 영화 리뷰)에 맞게 파인튜닝하고, 이를 글로벌 모델 허브인 Kaggle과 Hugging Face에 배포하는 전체 파이프라인을 구축하는 것을 목표로 합니다.

## 2. 주요 기술 스택
- **Framework**: Keras 3 (JAX Backend)
- **Library**: KerasHub, Hugging Face Hub, KaggleHub
- **Model**: GPT-2 (Causal Language Model)
- **Dataset**: IMDB Movie Reviews

## 3. 핵심 수행 내용

### ① 모델 파인튜닝 (Fine-tuning)
- `gpt2_base_en` 프리셋 모델을 로드하여 IMDB 데이터셋의 텍스트 데이터에 맞게 적응시켰습니다.
- `TopPSampler` 및 `Temperature` 설정을 통해 단순 반복을 피하고 자연스러운 문맥을 생성하도록 최적화했습니다.

### ② 멀티 허브 배포 (Multi-Hub Deployment)
- **Kaggle Models**: `keras_hub.upload_preset`을 통해 버전 관리(`v1`) 기능을 포함하여 배포했습니다.
- **Hugging Face Hub**: `hf://` URI 형식을 사용하여 모델을 업로드하고, 상세한 모델 카드(README.md)와 메타데이터를 작성하여 접근성을 높였습니다.

### ③ 유지보수 및 검증 (Validation & Utils)
- 배포된 모델과 로컬 모델 간의 **성능 비교 분석**을 수행하여 가중치 전송의 무결성을 확인했습니다.
- 모델 가중치를 안전하게 로드하기 위한 독립적인 유틸리티 모듈(`model_utils.py`)을 제작하여 코드의 재사용성을 확보했습니다.

## 4. 결과 및 성과
- **모델 접근성 확보**: API 호출만으로 누구나 `from_preset("hf://username/model_id")`을 통해 모델을 즉시 사용할 수 있는 환경을 구축했습니다.
- **문서화 최적화**: 학습 지표(Loss) 및 예제 생성 결과를 포함한 전문적인 모델 카드를 작성했습니다.

---

### ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
## 5. 파기파라긴 시가배 (Pipeline Visualization)

거배서 파기파라긴은 다으ᄆ가 가튼 흐르므로 구새돼어스바니다.

```mermaid
graph TD
    A[IMDB Dataset] -->|TFDS Load| B(Preprocessing)
    B -->|Text Extraction| C{GPT-2 Fine-tuning}
    D[KerasHub Preset] --> C
    C -->|1 Epoch Training| E[Fine-tuned Model]
    
    E -->|save_to_preset| F[Local Directory: ./my_finetuned_gpt2]
    
    F -->|upload_preset| G[Kaggle Model Hub]
    F -->|upload_preset| H[Hugging Face Hub]
    
    H -->|api.upload_file| I[README.md with Metrics]
    
    G -->|from_preset| J[Production Inference]
    H -->|from_preset| J
    
    style C fill:#f9f,stroke:#333,stroke-width:2px
    style F fill:#bbf,stroke:#333,stroke-width:2px
    style G fill:#dfd,stroke:#333,stroke-width:2px
    style H fill:#dfd,stroke:#333,stroke-width:2px
```

## 6. [Portfolio Summary] End-to-End LLM 파이프라인 구축 및 배포

### ┃ 기술적 핵심 요약 (Technical Highlights)
본 프로젝트는 데이터 전처리부터 모델 배포까지의 전 과정을 자동화된 워크플로우로 구현했습니다.

*   **아키텍처**: GPT-2 Base 모델에 Keras 3와 JAX 백엔드를 결합하여 고속 연산 및 효율적인 메모리 관리를 구현했습니다.
*   **데이터 파이프라인**: `tensorflow_datasets`를 활용해 IMDB 데이터를 로드하고, `map` 함수를 이용해 비정형 텍스트를 실시간으로 추출하여 모델에 공급하는 스트리밍 구조를 채택했습니다.
*   **모델 배포 전략**:
    *   **Kaggle**: 버전 관리를 통해 모델의 이력을 추적 가능하게 배포
    *   **Hugging Face**: `hf://` URI 스키마를 활용하여 모델 접근성을 극대화하고, YAML 프론트매터를 포함한 고도화된 모델 카드 작성

### ┃ 파이프라인 워크플로우 (Pipeline Workflow)

1.  **Preparation**: KerasHub 라이브러리 환경 구축 및 JAX 백엔드 가속 설정
2.  **Fine-tuning**: 전이 학습(Transfer Learning) 기법을 적용하여 일반 GPT-2를 영화 리뷰 특화 모델로 도메인 적응
3.  **Optimization**: `TopPSampler`를 도입하여 생성 텍스트의 다양성과 창의성을 확보
4.  **Serialization**: `save_to_preset`을 사용하여 토크나이저, 구성 파일, 가중치를 통합 패키징
5.  **Multi-Cloud Deployment**: 오픈소스 생태계(Hugging Face)와 데이터 사이언스 커뮤니티(Kaggle)에 동시 배포
6.  **Verification**: 원격 허브의 가중치를 직접 로드하여 로컬 결과와 교차 검증하는 QA 프로세스 수행

### ┃ 성과 및 기대 효과
*   **재사용성**: `model_utils.py` 모듈화를 통해 향후 다른 프로젝트에서도 동일한 로직으로 가중치를 안전하게 로드 가능
*   **확장성**: 동일한 파이프라인을 BERT, Llama 등 KerasHub가 지원하는 타 모델로 즉시 확장 가능
*   **문서화**: 커뮤니티 지향적인 모델 설명을 통해 사용자 경험(UX) 개선

### ┃ 주요 기술적 성과 요약 (Technical Achievement Summary)

| 구분 | 세부 항목 | 내용 |
| :--- | :--- | :--- |
| **Model** | Base Architecture | GPT-2 (Causal LM) |
| **Training** | Epochs / Loss | 1 Epoch / 1.0200 |
| **Backend** | Compute Engine | JAX Backend (Keras 3) |
| **Sampling** | Text Generation Strategy | Top-P (0.9), Temp (0.7) |
| **Deployment** | Multi-Hub Support | Kaggle (v1), Hugging Face Hub |
| **Automation** | Serialization | KerasHub Native Preset (.h5, .json) |
| **Quality** | Performance Verification | Local vs. Hub Consistency Analysis |

### ┃ 프로젝트 기술 아키텍처 (Technical Architecture Diagram)

아래는 본 프로젝트의 전체 워크플로우를 도식화한 다이어그램입니다.

```mermaid
graph LR
    subgraph "Data Phase"
    A[IMDB Raw Data] --> B{TFDS Pipeline}
    B --> C[Text Stream]
    end

    subgraph "Model Phase"
    D[GPT-2 Base] --> E{KerasHub Fine-tuning}
    C --> E
    E --> F[Optimized Model]
    end

    subgraph "Serialization"
    F --> G[save_to_preset]
    G --> H[Local Directory]
    end

    subgraph "Deployment Hubs"
    H --> I[Kaggle Models]
    H --> J[Hugging Face Hub]
    end

    subgraph "Verification"
    J --> K[Remote Load Test]
    I --> K
    end

    style E fill:#4CAF50,stroke:#333,color:#fff
    style H fill:#2196F3,stroke:#333,color:#fff
    style I fill:#FF9800,stroke:#333,color:#fff
    style J fill:#FF9800,stroke:#333,color:#fff
```

### 🏗️ Full Project Architecture (End-to-End Pipeline)

본 프로젝트의 전체 흐름을 요약한 통합 아키텍처 다이어그램입니다.

```mermaid
graph TD
    subgraph "1. Data & Fine-tuning"
    A[IMDB Dataset] --> B(TFDS Preprocessing)
    B --> C{GPT-2 Fine-tuning}
    D[KerasHub Preset] --> C
    C --> E[Teacher Model]
    end

    subgraph "2. Knowledge Distillation"
    E --> F{Distiller Class}
    G[Student Model] --> F
    F --> H[Distilled Student Model]
    end

    subgraph "3. Model Management & Hubs"
    E --> I[Local Save /my_finetuned_gpt2]
    H --> J[Local Save /distilled_gpt2_student]
    I --> K[Kaggle Models]
    I --> L[Hugging Face Hub]
    L --> M[Model Card: README.md]
    end

    subgraph "4. Deployment (Serving)"
    J --> N[Flask API Server]
    N --> O[ngrok Tunneling]
    O --> P[Public API URL]
    P --> Q[External User/Client]
    end

    style C fill:#f9f,stroke:#333,stroke-width:2px
    style F fill:#bbf,stroke:#333,stroke-width:2px
    style N fill:#dfd,stroke:#333,stroke-width:2px
    style P fill:#ffd,stroke:#333,stroke-width:2px
```

### 🏗️ Full Project Architecture (End-to-End Pipeline)

본 프로젝트의 전체 흐름을 요약한 통합 아키텍처 다이어그램입니다.

```mermaid
graph TD
    subgraph "1. Data & Fine-tuning"
    A[IMDB Dataset] --> B(TFDS Preprocessing)
    B --> C{GPT-2 Fine-tuning}
    D[KerasHub Preset] --> C
    C --> E[Teacher Model]
    end

    subgraph "2. Knowledge Distillation"
    E --> F{Distiller Class}
    G[Student Model] --> F
    F --> H[Distilled Student Model]
    end

    subgraph "3. Model Management & Hubs"
    E --> I[Local Save /my_finetuned_gpt2]
    H --> J[Local Save /distilled_gpt2_student]
    I --> K[Kaggle Models]
    I --> L[Hugging Face Hub]
    L --> M[Model Card: README.md]
    end

    subgraph "4. Deployment (Serving)"
    J --> N[Flask API Server]
    N --> O[ngrok Tunneling]
    O --> P[Public API URL]
    P --> Q[External User/Client]
    end

    style C fill:#f9f,stroke:#333,stroke-width:2px
    style F fill:#bbf,stroke:#333,stroke-width:2px
    style N fill:#dfd,stroke:#333,stroke-width:2px
    style P fill:#ffd,stroke:#333,stroke-width:2px
```

In [ ]:
portfolio_content = f"""# [Project Portfolio] GPT-2 Movie Review Generator

## 1. 프로젝트 개요
본 프로젝트는 GPT-2 기반의 LLM을 KerasHub를 사용하여 IMDB 영화 리뷰 데이터셋에 맞게 파인튜닝하고, 글로벌 모델 허브(Kaggle, Hugging Face)에 배포하는 전체 파이프라인을 구축한 사례입니다.

## 2. 주요 기술적 성과
| 구분 | 세부 항목 | 내용 |
| :--- | :--- | :--- |
| **Model** | Architecture | GPT-2 (Causal LM) |
| **Training** | Metrics | 1 Epoch / Loss: {training_loss} |
| **Backend** | Library | Keras 3 (JAX Backend) |
| **Sampling** | Strategy | Top-P (0.9), Temp (0.7) |
| **Deployment** | Hubs | Kaggle, Hugging Face |

## 3. 시스템 아키텍처
```mermaid
graph LR
    A[IMDB Data] --> B{{KerasHub Fine-tuning}}
    B --> C[Serialization]
    C --> D[Kaggle Hub]
    C --> E[Hugging Face Hub]
    E --> F[Verification & Test]
```

## 4. 핵심 워크플로우
1. **Data Pipeline**: `tensorflow_datasets`를 활용한 데이터 로드 및 정제.
2. **Model Optimization**: `TopPSampler` 적용을 통해 텍스트 생성의 창의성 및 품질 확보.
3. **Multi-Hub Strategy**: Kaggle 모델 버전 관리 및 Hugging Face 모델 카드 고도화.
4. **Robustness**: `model_utils.py`를 통한 가중치 로드 안정화 및 검증.

## 5. 최종 결과물
- **Hugging Face**: [https://huggingface.co/{hf_username}/{model_id}](https://huggingface.co/{hf_username}/{model_id})
- **Kaggle**: [https://www.kaggle.com/models/{kaggle_username}/{model_name}](https://www.kaggle.com/models/{kaggle_username}/{model_name})
"""

with open("Final_Portfolio.md", "w") as f:
    f.write(portfolio_content)

print("✅ 'Final_Portfolio.md' 파일이 성공적으로 생성되었습니다!")

In [ ]:
from google.colab import files

# 'Final_Portfolio.md' 파일을 로컬로 다운로드합니다.
files.download('Final_Portfolio.md')

In [ ]:
import shutil
from google.colab import files

# Define the name of the output zip file
zip_filename = 'gpt2_project_full_package'

# Create a temporary directory to collect all files
# Note: model directories (./my_finetuned_gpt2, ./distilled_gpt2_student) are already present

# Compress the current workspace (excluding some system folders if necessary)
# For simplicity, we will zip the specific project folders and files
!zip -r {zip_filename}.zip ./my_finetuned_gpt2 ./distilled_gpt2_student model_utils.py Final_Portfolio.md README*.md

print(f'Archive {zip_filename}.zip created successfully.')

# Download the file
files.download(f'{zip_filename}.zip')

In [ ]:
from google.colab import files
import os

file_path = 'gpt2_project_full_package.zip'
if os.path.exists(file_path):
    print(f'Triggering download for: {file_path}')
    files.download(file_path)
else:
    print(f'Error: {file_path} not found. Please run the compression cell again.')

## 📝 블로그 포스팅용 프로젝트 요약

### 🚀 KerasHub와 GPT-2를 활용한 영화 리뷰 생성 모델 구축 및 배포

이번 프로젝트에서는 **KerasHub**를 활용하여 GPT-2 모델을 파인튜닝하고, 이를 **Kaggle**과 **Hugging Face Hub**라는 글로벌 플랫폼에 배포하는 전체 엔드투엔드(End-to-End) 머신러닝 워크플로우를 완수했습니다.

#### 1. 주요 기술 스택
*   **Framework**: Keras 3 (JAX Backend)
*   **Model**: GPT-2 (Causal Language Model)
*   **Libraries**: KerasHub, KaggleHub, Hugging Face Hub
*   **Dataset**: IMDB Movie Reviews

#### 2. 핵심 워크플로우
1.  **데이터 준비**: IMDB 데이터셋을 로드하고 Causal LM 학습을 위해 텍스트 스트림으로 변환
2.  **파인튜닝**: `gpt2_base_en` 프리셋을 기반으로 도메인 특화 학습 수행 (최종 Loss: 1.0200)
3.  **최적화**: Top-P 샘플링 및 Temperature 설정을 통해 자연스럽고 다양한 문장 생성 구현
4.  **멀티 허브 배포**:
    *   **Kaggle**: 버전 관리 기능을 활용한 모델 저장
    *   **Hugging Face**: 상세한 모델 카드(README)와 함께 API 기반 배포
5.  **검증**: 원격 허브에서 모델을 다시 불러와 로컬 결과와 교차 검증 및 `model_utils.py` 모듈화

#### 3. 프로젝트 성과
*   **모델 접근성**: 단순한 URI 호출(`hf://...`)만으로 누구나 모델을 즉시 사용할 수 있는 환경 구축
*   **문서화 완비**: 성능 지표, 사용 예시, 아키텍처 다이어그램이 포함된 전문적인 모델 카드 작성
*   **재사용성**: 별도의 유틸리티 파일을 제작하여 향후 프로젝트에서의 확장성 확보

---
*이 포스트는 Google Colab 환경에서 KerasHub를 통해 제작된 기술 포트폴리오의 요약본입니다.*

### 함수를 이용한 모델 재로드 예시

정의된 `load_model_weights_safely` 함수를 사용하여 다운로드한 가중치를 새로운 모델 객체에 적용해 봅니다.

In [ ]:
def load_model_weights_safely(preset_name, weights_path, skip_mismatch=True):
    """
    KerasHub 모델의 구조를 생성하고, 가중치를 안전하게 로드합니다.
    """
    print(f"'{preset_name}' 구조로 모델 생성 중...")
    model = keras_hub.models.CausalLM.from_preset(preset_name)

    # 더미 데이터로 모델 빌드 (레이어 초기화)
    model.generate("warmup", max_length=2)

    try:
        # 백본에 가중치 로드
        model.backbone.load_weights(weights_path, skip_mismatch=skip_mismatch)
        print("✅ 가중치가 성공적으로 로드되었습니다.")
    except Exception as e:
        print(f"❌ 가중치 로드 실패: {e}")

    return model

# 함수를 호출하여 가중치가 적용된 새 모델 생성
reloaded_gpt2_model = load_model_weights_safely(
    preset_name="gpt2_base_en",
    weights_path=downloaded_path
)

# 샘플러 설정 (추론 품질 향상)
reloaded_gpt2_model.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))

# 최종 확인을 위한 텍스트 생성
print("\n--- 재로드된 모델의 생성 결과 ---")
print(reloaded_gpt2_model.generate("The cinematography in this film", max_length=50))

In [ ]:
model_utils_content = """
import keras_hub

def load_model_weights_safely(preset_name, weights_path, skip_mismatch=True):
    \"\"\"
    KerasHub 모델의 구조를 생성하고, 가중치를 안전하게 로드합니다.
    \"\"\"
    print(f"'{preset_name}' 구조로 모델 생성 중...")
    model = keras_hub.models.CausalLM.from_preset(preset_name)

    # 더미 데이터로 모델 빌드 (레이어 초기화)
    model.generate("warmup", max_length=2)

    try:
        # 백본에 가중치 로드
        model.backbone.load_weights(weights_path, skip_mismatch=skip_mismatch)
        print("✅ 가중치가 성공적으로 로드되었습니다.")
    except Exception as e:
        print(f"❌ 가중치 로드 실패: {e}")

    return model
"""

with open("model_utils.py", "w") as f:
    f.write(model_utils_content.strip())

print("✅ 'model_utils.py' 파일이 성공적으로 생성되었습니다.")

In [ ]:
from model_utils import load_model_weights_safely

# 1. 파일에서 임포트한 함수를 사용하여 모델 생성 및 가중치 로드
print("--- model_utils.py 테스트 시작 ---")
test_model_from_file = load_model_weights_safely(
    preset_name='gpt2_base_en',
    weights_path=downloaded_path
)

# 2. 샘플러 설정 및 추론 테스트
test_model_from_file.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))

print("\n--- 테스트 생성 결과 ---")
print(test_model_from_file.generate("Using a separate python file is", max_length=50))

print("\n✅ 'model_utils.py' 임포트 및 실행 테스트가 성공적으로 완료되었습니다!")

### Adding Advanced Metadata and Tags

You can add specialized tags like `library_name`, `pipeline_tag`, or custom tags to help users find your model. Refer to the Hugging Face documentation for a full list of supported metadata fields.

In [ ]:
updated_readme_content = """---
language: en
license: mit
tags:
- keras-hub
- gpt2
- text-generation
- movie-reviews
- natural-language-processing
- keras
datasets:
- imdb
library_name: keras-hub
pipeline_tag: text-generation
---

# GPT-2 Fine-tuned on IMDB

This model is a fine-tuned version of GPT-2 on movie reviews.

### Updated Metadata
- Added `pipeline_tag` for better UI integration on Hugging Face.
- Added more descriptive `tags` for search optimization.
"""

with open("README_updated.md", "w") as f:
    f.write(updated_readme_content)

# Upload the updated version
api.upload_file(
    path_or_fileobj="README_updated.md",
    path_in_repo="README.md",
    repo_id=f"{hf_username}/{model_id}",
    repo_type="model"
)

print("Model metadata and tags have been successfully updated!")

In [ ]:
final_readme_with_terms = """---
language: en
license: mit
tags:
- keras-hub
- gpt2
- text-generation
- movie-reviews
library_name: keras-hub
pipeline_tag: text-generation
---

# GPT-2 Fine-tuned on IMDB

This model is a fine-tuned version of GPT-2 on movie reviews.

## Terms of Use

By accessing this model, you agree to the following terms:
1. **Research Use Only**: This model is provided for educational and research purposes.
2. **No Malicious Use**: You may not use this model to generate harmful, illegal, or deceptive content.
3. **Attribution**: If you use this model in your work, please cite this repository.
4. **Disclaimer**: The model is provided 'as is' without any warranties.
"""

with open("README_final.md", "w") as f:
    f.write(final_readme_with_terms)

# Hugging Face Hub에 업로드
api.upload_file(
    path_or_fileobj="README_final.md",
    path_in_repo="README.md",
    repo_id=f"{hf_username}/{model_id}",
    repo_type="model"
)

print("약관이 포함된 README.md가 성공적으로 업데이트되었습니다!")

### 🧪 지식 증류(Knowledge Distillation) 실습 예제

이 예제는 간단한 분류 모델을 통해 Teacher 모델의 확률 분포를 Student 모델이 어떻게 학습하는지 보여주는 로직입니다.

In [ ]:
import keras
from keras import layers
import numpy as np
import tensorflow as tf

class Distiller(keras.Model):
    def __init__(self, student, teacher):
        super().__init__()
        self.teacher = teacher
        self.student = student

    def compile(self, optimizer, metrics, student_loss_fn, distillation_loss_fn, alpha=0.1, temperature=3):
        super().compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha
        self.temperature = temperature

    def train_step(self, data):
        # 텐서 언패킹 오류를 방지하기 위해 인덱싱 사용
        x = data
        y = data # Causal LM에서는 입력이 곧 타겟(Shifted internally)

        # Teacher 추론
        teacher_predictions = self.teacher(x, training=False)

        with keras.GradientTape() as tape:
            # Student 추론
            student_predictions = self.student(x, training=True)

            # 오차 계산
            student_loss = self.student_loss_fn(y, student_predictions)

            # 지식 증류 오차 (Temperature 적용)
            distillation_loss = self.distillation_loss_fn(
                keras.activations.softmax(teacher_predictions / self.temperature, axis=-1),
                keras.activations.softmax(student_predictions / self.temperature, axis=-1),
            )

            loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

        # 가중치 업데이트
        trainable_vars = self.student.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        # 지표 업데이트
        self.compiled_metrics.update_state(y, student_predictions)
        results = {m.name: m.result() for m in self.metrics}
        results.update({"student_loss": student_loss, "distillation_loss": distillation_loss})
        return results

### GPT-2 지식 증류(Knowledge Distillation) 실행 예제

1. **Teacher**: 이전에 파인튜닝하여 성능이 검증된 GPT-2 모델을 사용합니다.
2. **Student**: 지식을 전수받을 모델로, 여기서는 동일 구조의 초기화된 모델을 사용하거나 더 작은 모델을 선택할 수 있습니다.
3. **Distillation**: Teacher의 예측 확률(Soft Targets)을 Student가 학습하도록 유도합니다.

In [ ]:
import keras
import keras_hub

# 1. Teacher 모델 준비 (이미 파인튜닝된 로컬 프리셋 로드)
teacher_model = keras_hub.models.CausalLM.from_preset("./my_finetuned_gpt2")
teacher_model.trainable = False  # Teacher는 학습하지 않음

# 2. Student 모델 준비 (지식을 전수받을 초기 모델)
# 실제로는 레이어가 적은 더 작은 GPT-2 프리셋을 사용하면 좋습니다.
student_model = keras_hub.models.CausalLM.from_preset("gpt2_base_en")

# 3. Distiller 초기화
distiller = Distiller(student=student_model, teacher=teacher_model)

# 4. 컴파일 설정
# alpha: 정답(y)과 Teacher 예측값 사이의 비중 조절
# temperature: Soft Target의 확률 분포를 부드럽게 만들어주는 하이퍼파라미터
distiller.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=5e-5),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
    student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    distillation_loss_fn=keras.losses.KLDivergence(),
    alpha=0.1,
    temperature=5
)

# 5. 증류 학습 시작 (IMDB 데이터셋 활용)
# 데이터의 텍스트 부분만 추출하여 학습을 진행합니다.
distillation_data = imdb_train_texts.batch(2) # 메모리 효율을 위해 작은 배치 사이즈 사용

print("🚀 지식 증류 학습을 시작합니다...")
history = distiller.fit(distillation_data, epochs=1)

print("\n✅ 학습 완료! 이제 Student 모델은 Teacher의 지식을 모방하도록 훈련되었습니다.")

### 학습 결과 확인
증류 학습이 끝난 Student 모델의 텍스트 생성 능력을 테스트하여 Teacher의 스타일을 얼마나 잘 복사했는지 확인해 봅니다.

In [ ]:
prompt = "The movie was"
student_output = student_model.generate(prompt, max_length=50)

print("--- Student Model (Distilled) Output ---")
print(student_output)

### Teacher vs Student 예측 분포 비교 시각화

지식 증류의 핵심은 Student가 Teacher의 확률 분포(Soft Targets)를 배우는 것입니다. 특정 단어 뒤에 올 다음 단어들의 확률을 시각화하여 비교해 봅니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

def get_top_logits(model, prompt, top_k=10):
    # 1. 전처리 수행
    processed = model.preprocessor(prompt)

    # KerasHub preprocessor can return a tuple (inputs, labels) or just inputs (dict)
    if isinstance(processed, tuple):
        processed = processed[0]

    # 2. 텐서 타입 정규화 및 배치 차원 추가
    inputs = {}
    if isinstance(processed, dict):
        for k, v in processed.items():
            tensor = tf.convert_to_tensor(v)
            if tensor.dtype == tf.bool or tensor.dtype.is_integer:
                inputs[k] = tf.expand_dims(tf.cast(tensor, tf.int32), 0)
            else:
                inputs[k] = tf.expand_dims(tensor, 0)
    else:
        inputs = tf.expand_dims(tf.cast(processed, tf.int32), 0)

    # 3. 모델 호출 (추론)
    outputs = model(inputs, training=False)

    # 4. 마지막 토큰의 로짓(Logits) 추출
    last_logits = outputs[0, -1, :]

    # 5. 상위 K개 결과 추출
    logits_numpy = last_logits.numpy()
    top_indices = np.argsort(logits_numpy)[-top_k:][::-1]
    top_values = logits_numpy[top_indices]

    # 6. 토큰 ID를 단어로 변환
    tokenizer = model.preprocessor.tokenizer
    top_words = [tokenizer.id_to_token(i) for i in top_indices]

    return top_words, top_values

# 실행 및 그래프 출력
test_prompt = "The movie was"
top_k = 10

try:
    # 데이터 확보
    words, teacher_logits = get_top_logits(teacher_model, test_prompt, top_k=top_k)
    _, student_logits = get_top_logits(student_model, test_prompt, top_k=top_k)

    # 확률 분포 계산 (Softmax)
    def softmax(x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum()

    teacher_probs = softmax(teacher_logits)
    student_probs = softmax(student_logits)

    # 시각화 설정
    x = np.arange(len(words))
    width = 0.35
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.bar(x - width/2, teacher_probs, width, label='Teacher (Fine-tuned)', color='#4A90E2', alpha=0.8)
    ax.bar(x + width/2, student_probs, width, label='Student (Distilled)', color='#F5A623', alpha=0.8)

    ax.set_ylabel('Probability')
    ax.set_title(f'Next Token Prediction: Teacher vs Student\nPrompt: "{test_prompt}"')
    ax.set_xticks(x)
    ax.set_xticklabels(words, rotation=45)
    ax.legend()

    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

except Exception as e:
    import traceback
    print(f"시각화 중 오류 발생:\n{e}")
    traceback.print_exc()

### Student 모델 로컬 저장

지식 증류(Knowledge Distillation)를 통해 Teacher의 성능을 이어받은 Student 모델을 나중에 다시 사용하거나 배포할 수 있도록 로컬 프리셋으로 저장합니다.

In [ ]:
# Student 모델 저장 경로 설정
student_preset_dir = "./distilled_gpt2_student"

# 모델 저장
student_model.save_to_preset(student_preset_dir)

print(f"✅ Student 모델이 '{student_preset_dir}' 경로에 성공적으로 저장되었습니다.")
# 저장된 파일 목록 확인
import os
print("저장된 파일 목록:", os.listdir(student_preset_dir))

### 저장된 Student 모델을 활용한 추론 테스트

로컬에 저장된 프리셋을 불러와 실제 텍스트 생성 성능을 확인합니다.

In [ ]:
import keras_hub

# 1. 저장된 경로에서 Student 모델 로드
reloaded_student = keras_hub.models.CausalLM.from_preset("./distilled_gpt2_student")

# 2. 텍스트 품질 향상을 위한 샘플러 설정 (Top-P Sampling)
reloaded_student.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))

# 3. 추론 실행
test_prompt = "I thought this movie was"
print(f"Prompt: {test_prompt}")

generated_text = reloaded_student.generate(test_prompt, max_length=80)

print("\n--- Generated Review (Student Model) ---")
print(generated_text)

### Student 모델 웹 API 서비스 예제 (Flask)

학습된 모델을 실제 애플리케이션에서 호출할 수 있도록 Flask를 사용하여 간단한 추론 서버를 구축합니다.

In [ ]:
!pip install -q Flask flask-cors pyngrok requests

In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import os

# 1. ngrok Authtoken 설정 확인 및 적용
try:
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    if not auth_token:
        raise ValueError("NGROK_AUTH_TOKEN이 설정되지 않았습니다. 왼쪽 '열쇠' 메뉴에서 설정해 주세요.")

    # 기존 프로세스 종료 (재시도 시 충돌 방지)
    ngrok.kill()

    ngrok.set_auth_token(auth_token)

    # 2. 터널 열기 (Flask 기본 포트인 5000번)
    public_url = ngrok.connect(5000)
    print(f"* 외부 접속 가능 URL: {public_url}")
    print(f"* API 테스트 주소: {public_url}/generate")

except Exception as e:
    print(f"오류 발생: {e}")
    print("팁: Secrets 관리자에서 NGROK_AUTH_TOKEN의 'Notebook access'가 켜져 있는지 확인해 주세요.")

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import sys

def setup_ngrok_tunnel(port=5000):
    try:
        # 1. Colab Secrets에서 토큰 가져오기
        auth_token = userdata.get('NGROK_AUTH_TOKEN')
        if not auth_token:
            raise ValueError("Secrets에서 NGROK_AUTH_TOKEN을 찾을 수 없습니다.")

        # 2. ngrok 인증 및 기존 터널 정리
        ngrok.set_auth_token(auth_token)
        ngrok.kill()

        # 3. 새로운 터널 생성
        public_url = ngrok.connect(port)

        print(f"✅ ngrok 터널 생성 성공!")
        print(f"🔗 외부 접속 URL: {public_url}")
        print(f"📝 API 테스트: {public_url}/generate")
        return public_url

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        print("\n[해결 방법]")
        print("Colab 왼쪽 '열쇠' 아이콘 메뉴에서 NGROK_AUTH_TOKEN을 추가하고 'Notebook access'를 켰는지 확인하세요.")
        return None

# 터널 실행
public_url = setup_ngrok_tunnel(5000)

In [ ]:
from google.colab import userdata

try:
    # 'secretName' 대신 실제 Secrets에 저장한 'NGROK_AUTH_TOKEN'을 사용해야 합니다.
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    print("성공적으로 토큰을 불러왔습니다.")
except Exception as e:
    print(f"오류 발생: {e}")
    print("팁: 왼쪽 '열쇠' 아이콘 메뉴에서 'NGROK_AUTH_TOKEN'이 추가되어 있는지 확인하세요.")

In [ ]:
try:
    from flask import Flask, request, jsonify
    from flask_cors import CORS
    import keras_hub
    import threading
    import time
    import os

    # 1. 앱 설정
    app = Flask(__name__)
    CORS(app)

    # 2. 모델 경로 복구 (없을 경우 HF에서 로드)
    model_path = "./my_finetuned_gpt2"
    if not os.path.exists(model_path):
        print(f"📦 로컬 모델을 찾을 수 없습니다. Hugging Face Hub에서 다시 로드합니다...")
        # 이전 세션에서 성공적으로 업로드된 URI 사용
        hf_uri = "hf://hinttt/gpt2-imdb-finetuned"
        api_model = keras_hub.models.CausalLM.from_preset(hf_uri)
        # 로컬 백업 생성
        api_model.save_to_preset(model_path)
        print(f"✅ 모델이 {model_path}에 복구되었습니다.")
    else:
        print(f"🚀 기존 로컬 모델을 로드합니다: {model_path}")
        api_model = keras_hub.models.CausalLM.from_preset(model_path)

    api_model.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))
    print("✅ Model compilation successful.")

    @app.route("/generate", methods=["POST"])
    def generate_text():
        data = request.json
        prompt = data.get("prompt", "This movie was")
        max_length = data.get("max_length", 80)
        output = api_model.generate(prompt, max_length=max_length)
        return jsonify({"prompt": prompt, "generated_text": output})

    @app.route("/", methods=["GET"])
    def index():
        return "Model API is Running!"

    def run_app():
        app.run(host='0.0.0.0', port=5000, use_reloader=False)

    # 백그라운드 스레드에서 서버 시작
    server_thread = threading.Thread(target=run_app, daemon=True)
    server_thread.start()
    time.sleep(5)
    print("\n🚀 API Server started on port 5000 in background.")

except Exception as e:
    print(f"❌ 오류 발생: {e}")

In [ ]:
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS
import keras_hub
import time
import os
import subprocess

def kill_port_5000():
    try:
        result = subprocess.check_output(["lsof", "-t", "-i:5000"]).decode().strip()
        if result:
            for pid in result.split("\n"):
                os.system(f"kill -9 {pid}")
            print(f"✅ 기존 5000번 포트 프로세스({result})를 종료했습니다.")
    except:
        pass

kill_port_5000()

inference_lock = threading.Lock()
app = Flask(__name__)
CORS(app)

# 모델 로드 (전역 변수)
model_path = "./my_finetuned_gpt2"
print("⏳ 모델 로딩 중...")
if not os.path.exists(model_path):
    api_model = keras_hub.models.CausalLM.from_preset("hf://hinttt/gpt2-imdb-finetuned")
else:
    api_model = keras_hub.models.CausalLM.from_preset(model_path)

api_model.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))
print("✅ 모델 로드 및 컴파일 완료!")

@app.route("/generate", methods=["POST"])
def generate_text():
    try:
        data = request.json
        prompt = data.get("prompt", "This movie was")
        max_length = int(data.get("max_length", 80))
        with inference_lock:
            output = api_model.generate(prompt, max_length=max_length)
        return jsonify({"status": "success", "generated_text": output})
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 500

@app.route("/health", methods=["GET"])
def health_check():
    return jsonify({"status": "healthy", "time": time.time()})

def start_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# 데몬 스레드로 서버 시작
server_thread = threading.Thread(target=start_flask, daemon=True)
server_thread.start()
print("🚀 Flask 서버가 백그라운드(Port 5000)에서 시작되었습니다.")

### API 테스트 방법
서버가 가동된 후, 아래 셀을 실행하여 로컬에서 API가 정상 작동하는지 확인할 수 있습니다.

In [ ]:
import requests

test_url = "http://127.0.0.1:5000/generate"
data = {"prompt": "The director of this film", "max_length": 50}

response = requests.post(test_url, json=data)
print("API Response:")
print(response.json())

In [ ]:
import requests

# Flask 서버 로컬 주소
local_url = "http://127.0.0.1:5000/generate"

# 테스트용 페이로드
test_payload = {
    "prompt": "The acting in this movie was",
    "max_length": 50
}

try:
    print(f"📡 로컬 서버({local_url})에 요청을 보냅니다 (최대 60초 대기)...")
    # 첫 실행 시 모델 로드 부하를 고려하여 timeout을 60초로 상향
    response = requests.post(local_url, json=test_payload, timeout=60)

    if response.status_code == 200:
        print("✅ API 응답 성공!")
        print("--- 생성 결과 ---")
        print(response.json().get('generated_text'))
    else:
        print(f"❌ 서버 오류 (상태 코드: {response.status_code})")
        print(response.text)
except Exception as e:
    print(f"❌ 연결 실패 또는 타임아웃: {e}")
    print("💡 팁: Flask 서버 셀(24e82065)에서 모델 로드가 완료되었는지(✅) 확인해 주세요.")

In [ ]:
import concurrent.futures
import requests
import time

# 1. 설정
BASE_URL = "http://127.0.0.1:5000"
API_URL = f"{BASE_URL}/generate"
HEALTH_URL = f"{BASE_URL}/health"
CONCURRENT_REQUESTS = 5
PROMPT_LIST = [
    "The acting was",
    "The plot is",
    "I really liked",
    "One major issue",
    "Overall, this film"
]

# 2. 서버 가동 여부 확인 (Health Check)
def wait_for_server(attempts=10):
    print("🔍 서버 응답 확인 중...")
    for i in range(attempts):
        try:
            response = requests.get(HEALTH_URL, timeout=5)
            if response.status_code == 200:
                print("✅ 서버가 활성화되었습니다.")
                return True
        except:
            pass
        print(f"⏳ 대기 중... ({i+1}/{attempts})")
        time.sleep(2)
    return False

def send_request(prompt):
    payload = {"prompt": prompt, "max_length": 50}
    start = time.time()
    try:
        # 모델 추론 시간에 따라 timeout을 넉넉히 설정
        response = requests.post(API_URL, json=payload, timeout=120)
        duration = time.time() - start
        if response.status_code == 200:
            return f"[성공] '{prompt}' -> {duration:.2f}s 소요"
        else:
            return f"[실패] 상태 코드: {response.status_code}"
    except Exception as e:
        return f"[에러] {str(e)}"

if wait_for_server():
    print(f"🔥 {CONCURRENT_REQUESTS}개의 동시 요청 부하 테스트를 시작합니다...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENT_REQUESTS) as executor:
        results = list(executor.map(send_request, PROMPT_LIST[:CONCURRENT_REQUESTS]))

    print("\n--- 부하 테스트 결과 ---")
    for res in results:
        print(res)
else:
    print("❌ 서버 연결 실패. Flask 서버 셀의 로그를 확인해 주세요.")

### 📊 API 서버 실시간 로그 모니터링
백그라운드에서 실행되는 Flask 서버의 요청 처리 상태와 에러 발생 여부를 실시간으로 모니터링합니다.

In [ ]:
import logging
import sys

# 1. Flask 및 Werkzeug 로그 설정 변경
# 표준 출력을 가로채서 이 셀의 아웃풋에 나타나도록 합니다.
log = logging.getLogger('werkzeug')
log.setLevel(logging.INFO)

# 기존 핸들러 제거 후 새로운 스트림 핸들러 추가
if log.handlers:
    for handler in log.handlers:
        log.removeHandler(handler)

stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('[%(asctime)s] %(message)s')
stream_handler.setFormatter(formatter)
log.addHandler(stream_handler)

print("✅ 이제부터 서버에 들어오는 모든 요청 로그가 이 셀 아래에 실시간으로 표시됩니다.")
print("테스트를 위해 부하 테스트 셀이나 브라우저에서 API를 호출해 보세요.\n")

In [ ]:
from google.colab import userdata

try:
    # 'secretName' 대신 실제 Secrets에 저장한 'NGROK_AUTH_TOKEN'을 호출합니다.
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ 'NGROK_AUTH_TOKEN'을 성공적으로 불러왔습니다!")
except Exception as e:
    print(f"❌ 오류 발생: {e}")
    print("💡 해결 방법: 왼쪽 '열쇠' 아이콘 메뉴에서 'NGROK_AUTH_TOKEN'을 추가하고 'Notebook access'를 켜주세요.")

In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [ ]:
from google.colab import userdata

try:
    # 'secretName' 대신 실제 등록한 이름인 'NGROK_AUTH_TOKEN'을 사용합니다.
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ 'NGROK_AUTH_TOKEN'을 성공적으로 불러왔습니다!")
except userdata.SecretNotFoundError:
    print("❌ 오류: 'NGROK_AUTH_TOKEN' 시크릿을 찾을 수 없습니다.")
except userdata.NotebookAccessError:
    print("❌ 오류: 시크릿은 있지만 'Notebook access' 권한이 꺼져 있습니다.")

### 🌐 ngrok 터널 활성화 및 API 서버 최종 연결

이 단계에서는 `NGROK_AUTH_TOKEN`을 사용하여 로컬 Flask 서버를 외부 URL로 노출합니다.
1. 왼쪽 **Secrets(열쇠 아이콘)** 메뉴에서 `NGROK_AUTH_TOKEN` 추가 확인
2. **Notebook access** 권한 스위치 ON

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
from google.colab import userdata

try:
    # 1. 토큰 가져오기
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(auth_token)

    # 2. 기존 터널 종료 (충돌 방지)
    ngrok.kill()

    # 3. Flask 포트(5000) 연결
    public_url = ngrok.connect(5000)

    print("✅ ngrok 터널 생성 성공!")
    print(f"🔗 외부 접속 가능 URL: {public_url}")
    print(f"📝 API 테스트 엔드포인트: {public_url}/generate")

except Exception as e:
    print(f"❌ 오류 발생: {e}")

### 🧪 ngrok 외부 URL을 통한 API 테스트
위에서 생성된 `public_url`을 사용하여 실제로 외부 통신이 가능한지 확인합니다.

In [ ]:
import requests
import json

# 현재 활성화된 ngrok URL (셀 52485bd3에서 생성된 최신 주소)
NGROK_URL = "https://b555-8-228-12-166.ngrok-free.app"

test_endpoint = f"{NGROK_URL}/generate"
payload = {
    "prompt": "The movie was a masterpiece because",
    "max_length": 50
}

print(f"📡 외부 URL로 요청 전송 중: {test_endpoint}...")
try:
    # ngrok 브라우저 경고를 건너뛰기 위한 헤더 추가
    headers = {'ngrok-skip-browser-warning': 'true'}
    response = requests.post(test_endpoint, json=payload, headers=headers, timeout=60)

    if response.status_code == 200:
        print("✅ 외부 API 접속 성공!")
        print("--- 생성된 리뷰 ---")
        print(response.json().get('generated_text'))
    else:
        print(f"❌ 서버 응답 오류: {response.status_code}")
        print("💡 해결 방법: Flask 서버 셀(12e86850)이 현재 실행 상태인지 확인해 주세요.")
except Exception as e:
    print(f"❌ 연결 실패: {e}")

### 🔍 생성된 텍스트 품질 분석 및 검증
다양한 프롬프트를 사용하여 모델이 영화 리뷰 특유의 어조(톤앤매너)를 잘 구현하는지 테스트합니다.

In [ ]:
import requests

def check_generation_quality(prompt, max_len=80):
    url = "https://b555-8-228-12-166.ngrok-free.app/generate"
    headers = {'ngrok-skip-browser-warning': 'true'}
    payload = {"prompt": prompt, "max_length": max_len}

    print(f"Prompt: '{prompt}'")
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=60)
        if response.status_code == 200:
            result = response.json().get('generated_text', '')
            print(f"\n--- Generated Text ---\n{result}\n")
            print("="*50)
        else:
            print(f"❌ Error: {response.status_code}")
    except Exception as e:
        print(f"❌ Request failed: {e}")

# 다양한 상황의 프롬프트로 테스트
test_prompts = [
    "I absolutely loved this movie because",
    "The main problem with this film is",
    "If you are looking for a great horror movie,"
]

for p in test_prompts:
    check_generation_quality(p)

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

try:
    # 1. Secret 로드 테스트
    token = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ Secret 로드 성공!")

    # 2. ngrok 설정 및 연결
    ngrok.kill()
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(5000)

    print(f"🚀 서버가 외부로 연결되었습니다!")
    print(f"🔗 접속 주소: {public_url}")

except Exception as e:
    print(f"❌ 여전히 오류가 발생합니다: {e}")
    print("💡 해결 방법: 왼쪽 열쇠 아이콘 -> 'NGROK_AUTH_TOKEN' 추가 -> 'Notebook access' 활성화")

In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [ ]:
import os
from google.colab import userdata

try:
    # 현재 접근 가능한 모든 시크릿 키 목록 가져오기
    # (보안을 위해 값은 출력하지 않습니다)
    print("🔍 현재 접근 가능한 시크릿 목록:")
    # userdata 객체 내부의 키들을 확인하는 우회적인 방법
    # 실제로는 설정된 키가 없으면 KeyError가 발생하므로 직접 확인이 필요합니다.

    token = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ 'NGROK_AUTH_TOKEN'을 찾았습니다!")
except Exception as e:
    print(f"❌ 'NGROK_AUTH_TOKEN'을 찾을 수 없습니다.")
    print("\n[체크리스트]")
    print("1. 왼쪽 열쇠 아이콘(Secrets) 클릭")
    print("2. Name이 정확히 'NGROK_AUTH_TOKEN'인지 확인 (공백 포함 여부 등)")
    print("3. 해당 이름 바로 옆의 'Notebook access' 스위치를 켰는지 확인 (파란색 상태)")

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import threading

try:
    # 1. 토큰 가져오기 (Secrets 설정 확인)
    token = userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(token)

    # 2. 기존 터널 종료 후 새 터널 생성
    ngrok.kill()
    public_url = ngrok.connect(5000)

    print("✅ ngrok 터널 생성 성공!")
    print(f"🔗 외부 접속 URL: {public_url}")
    print(f"📝 API 테스트: {public_url}/generate")

except Exception as e:
    print(f"❌ 오류 발생: {e}")
    print("\n💡 [필독] 해결 방법:")
    print("1. 왼쪽 '열쇠' 아이콘 클릭")
    print("2. 'NGROK_AUTH_TOKEN' 추가 및 값 입력")
    print("3. 'Notebook access' 스위치를 반드시 켬")

### 🌐 ngrok 터널 활성화 및 외부 접속 주소 생성

In [ ]:
from google.colab import userdata
import sys

try:
    # 1. 시크릿 존재 여부 및 액세스 권한 확인
    token = userdata.get('NGROK_AUTH_TOKEN')

    # 2. 토큰 형식 간단 검증 (보안을 위해 일부만 출력)
    if token:
        print("✅ [확인 완료] NGROK_AUTH_TOKEN이 정상적으로 등록되었습니다.")
        print(f"ℹ️ 토큰 앞글자 확인: {token[:5]}***")
    else:
        print("⚠️ 경고: 토큰 값이 비어있습니다.")

except userdata.SecretNotFoundError:
    print("❌ 오류: 'NGROK_AUTH_TOKEN'이라는 이름의 시크릿을 찾을 수 없습니다.")
    print("💡 해결: 왼쪽 열쇠 아이콘 클릭 -> [Add new secret] -> Name에 'NGROK_AUTH_TOKEN' 입력")

except userdata.NotebookAccessError:
    print("❌ 오류: 시크릿은 존재하지만 'Notebook access' 권한이 없습니다.")
    print("💡 해결: 왼쪽 열쇠 아이콘 클릭 -> 'NGROK_AUTH_TOKEN' 항목 옆의 스위치를 파란색(ON)으로 변경")

except Exception as e:
    print(f"❌ 기타 오류 발생: {e}")

In [ ]:
from google.colab import userdata

try:
    # 'secretName' 대신 실제 등록한 이름인 'NGROK_AUTH_TOKEN'을 사용합니다.
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ 'NGROK_AUTH_TOKEN' 시크릿을 성공적으로 가져왔습니다!")
except userdata.SecretNotFoundError:
    print("❌ 오류: 'NGROK_AUTH_TOKEN'이라는 이름의 시크릿을 찾을 수 없습니다.")
    print("💡 해결: 왼쪽 열쇠 아이콘 클릭 -> [Add new secret] -> Name에 'NGROK_AUTH_TOKEN' 입력")
except userdata.NotebookAccessError:
    print("❌ 오류: 시크릿은 존재하지만 'Notebook access' 권한이 없습니다.")

### 모델 요약 및 설명 개선 (Final Version)

모델의 성능과 사용법을 더 명확하게 전달하기 위해 `README.md`의 내용을 최종적으로 다듬어 업데이트합니다.

### 🔍 ngrok 터널 상태 진단 및 디버깅
연결 오류가 발생할 경우 아래 코드를 통해 활성화된 터널 목록을 확인하고 설정을 초기화할 수 있습니다.

In [ ]:
import requests
from pyngrok import ngrok

print("🔍 [진단 1] ngrok 터널 상태 확인")
tunnels = ngrok.get_tunnels()
for t in tunnels:
    print(f"- 활성 터널: {t.public_url} -> {t.config['addr']}")

print("\n🔍 [진단 2] 로컬 Flask 서버(Port 5000) 응답 확인")
try:
    # 로컬에서 직접 호출하여 서버가 살아있는지 확인
    local_test = requests.get('http://127.0.0.1:5000/', timeout=2)
    print(f"✅ 로컬 서버 응답 성공: {local_test.text}")
except Exception as e:
    print(f"❌ 로컬 서버 응답 없음: {e}")
    print("\n💡 해결 방법: Flask 서버를 정의한 셀(cell_id: 24e82065)로 올라가서 해당 셀을 다시 실행해 주세요.")
    print("서버가 실행된 후 'API Server started on port 5000.' 메시지가 나오는지 확인이 필요합니다.")

In [ ]:
import os
from pyngrok import ngrok

# 1. 모든 기존 ngrok 프로세스 강제 종료
print("🔄 ngrok 프로세스를 초기화합니다...")
ngrok.kill()

# 2. 캐시된 설정 파일이 있다면 제거 (필요 시)
# !rm -rf /root/.ngrok2

print("✅ 초기화 완료. 이제 Secrets에서 토큰을 수정한 후 다시 연결 셀을 실행하세요.")

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

try:
    # 1. 업데이트된 토큰 가져오기
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(auth_token)

    # 2. Flask 포트(5000)에 다시 연결
    public_url = ngrok.connect(5000)

    print("✅ ngrok 터널 재연결 성공!")
    print(f"🔗 새로운 외부 접속 URL: {public_url}")
    print(f"📝 이 주소를 복사하여 아래 테스트 셀의 NGROK_URL에 입력하세요.")

except Exception as e:
    print(f"❌ 연결 실패: {e}")
    print("💡 팁: 토큰을 수정한 후에도 오류가 난다면 메뉴의 [런타임] -> [세션 다시 시작] 후 처음부터 실행해 보세요.")

In [ ]:
from pyngrok import ngrok
import requests

print("=== ngrok 터널 상태 확인 ===")
tunnels = ngrok.get_tunnels()
if not tunnels:
    print("❌ 현재 활성화된 터널이 없습니다.")
else:
    for tunnel in tunnels:
        print(f"📌 Public URL: {tunnel.public_url}")
        print(f"🔌 Local Addr: {tunnel.config['addr']}")
        print(f"📡 Protocol: {tunnel.proto}")

print("\n=== ngrok 로컬 진단 API 로그 (최근 5건) ===")
try:
    # ngrok 에이전트가 로컬에서 제공하는 상태 API 호출
    stats = requests.get("http://localhost:4040/api/requests/http?limit=5").json()
    for req in stats.get('requests', []):
        print(f"[{req['start']}] {req['request']['method']} {req['request']['uri']} -> {req['response']['status_code'] if req['response'] else 'No Response'}")
except Exception as e:
    print(f"⚠️ 진단 API에 접속할 수 없습니다: {e}")

### 🛠️ 502 Bad Gateway 원인 진단 및 해결 방법

502 에러는 보통 다음 상황에서 발생합니다:
1. **Flask 서버 종료**: 코드가 에러로 멈췄거나 런타임이 재시작됨.
2. **포트 불일치**: Flask는 5000번인데 ngrok은 다른 포트를 보고 있음.
3. **추론 중 충돌**: 모델 추론 중 메모리 부족(OOM)으로 프로세스가 죽음.

In [ ]:
import requests

def diagnose_502():
    print("🔍 [1단계] 로컬 Flask 서버 가동 여부 확인...")
    try:
        # 로컬 헬스체크 엔드포인트 호출
        res = requests.get("http://127.0.0.1:5000/health", timeout=2)
        print(f"✅ Flask 서버 응답 성공: {res.json()}")
    except Exception as e:
        print(f"❌ Flask 서버가 응답하지 않습니다: {e}")
        print("\n💡 [해결책] 'cell_id: 12e86850' (Flask 서버 시작 셀)을 다시 실행해 주세요.")

    print("\n🔍 [2단계] 포트 점유 상태 확인...")
    !lsof -i:5000 || echo "⚠️ 5000번 포트에서 실행 중인 프로세스가 없습니다."

diagnose_502()

### 🔄 ngrok 터널 재시작 및 502 에러 조치
기존 터널을 강제 종료하고 새로운 연결을 생성하여 게이트웨이 이슈를 해결합니다.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import time

try:
    print("1. 기존 ngrok 프로세스 및 터널 종료 중...")
    ngrok.kill()
    time.sleep(2)

    print("2. Secrets에서 NGROK_AUTH_TOKEN 로드 중...")
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(auth_token)

    print("3. Flask 서버(Port 5000)에 새 터널 연결 중...")
    new_public_url = ngrok.connect(5000)

    print(f"\n✅ ngrok 터널 재시작 성공!")
    print(f"🔗 새로운 외부 URL: {new_public_url}")
    print(f"📝 위 주소를 복사하여 테스트에 사용하세요.")

except Exception as e:
    print(f"\n❌ 터널 재시작 실패: {e}")
    print("💡 해결 팁: 'NGROK_AUTH_TOKEN'이 올바른지, 그리고 Flask 서버가 현재 실행 중인지 확인하세요.")

만약 `ERR_NGROK_105` (유효하지 않은 토큰) 오류가 계속된다면, [ngrok Dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)에서 토큰을 다시 복사하여 Secrets에 업데이트한 후 런타임을 다시 시작하는 것이 좋습니다.

In [ ]:
improved_readme = f"""---
language: en
license: mit
tags:
- keras-hub
- gpt2
- text-generation
- movie-reviews
- fine-tuned
library_name: keras-hub
pipeline_tag: text-generation
---

# GPT-2 Movie Review Generator (Fine-tuned via KerasHub)

이 모델은 **KerasHub**를 사용하여 IMDB 영화 리뷰 데이터셋으로 파인튜닝된 GPT-2 기반 텍스트 생성 모델입니다.

## 주요 특징
- **Base Model:** GPT-2 Base
- **Fine-tuning Dataset:** IMDB Reviews (Movie critiques)
- **Training Engine:** Keras 3 with JAX Backend
- **Capabilities:** 영화의 분위기와 비평가의 어조를 반영한 자연스러운 리뷰 생성

## 추천 사용법 (Inference)

최적화된 텍스트 생성을 위해 `TopPSampler` 사용을 권장합니다:

```python
import keras_hub

model = keras_hub.models.CausalLM.from_preset(\"hf://{hf_username}/{model_id}\")
model.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))

print(model.generate(\"This movie was\", max_length=80))
```

## 이용 약관
본 모델은 연구 및 교육용으로 제공되며, 악의적인 용도로의 사용을 금지합니다.
"""

with open("README_improved.md", "w") as f:
    f.write(improved_readme)

# Hugging Face Hub 업데이트
api.upload_file(
    path_or_fileobj="README_improved.md",
    path_in_repo="README.md",
    repo_id=f"{hf_username}/{model_id}",
    repo_type="model"
)

print("개선된 모델 요약이 반영되었습니다!")

### Adding Performance Metrics and Examples to README

To make the model card even better, let's include the loss observed during training and some sample outputs generated by the model.

In [ ]:
import huggingface_hub
from huggingface_hub import HfApi

api = HfApi()
hf_username = huggingface_hub.whoami()["name"]
model_id = "gpt2-imdb-finetuned"

# Define performance and examples
training_loss = "1.0200"
example_prompt = "This movie was"
example_output = "one of the first I saw in theaters and I was so excited to see it again..."

readme_with_metrics = f"""---
language: en
license: mit
tags:
- keras-hub
- gpt2
- text-generation
- movie-reviews
library_name: keras-hub
pipeline_tag: text-generation
---

# GPT-2 Movie Review Generator

## Performance Metrics
- **Training Loss:** {training_loss} (after 1 epoch)
- **Hardware:** Google Colab GPU/TPU

## Example Generation
**Prompt:** `{example_prompt}`

**Output:**
> {example_output}

## How to use
```python
import keras_hub
model = keras_hub.models.CausalLM.from_preset(\"hf://{hf_username}/{model_id}\")
output = model.generate(\"{example_prompt}\", max_length=100)
```
"""

with open("README_metrics.md", "w") as f:
    f.write(readme_with_metrics)

# Upload to Hugging Face
api.upload_file(
    path_or_fileobj="README_metrics.md",
    path_in_repo="README.md",
    repo_id=f"{hf_username}/{model_id}",
    repo_type="model"
)

print("README updated with metrics and examples!")

In [ ]:
import huggingface_hub
import keras_hub

try:
    # 로그인 정보 확인
    user_info = huggingface_hub.whoami()
    hf_username = user_info['name']

    # 새로운 버전의 모델 ID 설정
    new_model_id = "gpt2-imdb-finetuned-v2"
    hf_uri = f"hf://{hf_username}/{new_model_id}"

    # 로컬 프리셋 디렉토리 경로
    local_preset_dir = "./my_finetuned_gpt2"

    print(f"새로운 버전 업로드 중: {hf_uri}...")
    # Hugging Face Hub로 업로드
    keras_hub.upload_preset(hf_uri, local_preset_dir)

    print(f"\n✨ 새로운 버전이 성공적으로 업로드되었습니다!")
    print(f"주소: https://huggingface.co/{hf_username}/{new_model_id}")

except Exception as e:
    print(f"업로드 실패: {e}")

In [ ]:
import keras_hub

# 1. Hugging Face URI 설정
# 이전 단계에서 업로드한 모델의 URI를 사용합니다.
hf_model_uri = f"hf://{hf_username}/gpt2-imdb-finetuned"

print(f"Hugging Face Hub에서 모델 로드 중: {hf_model_uri}...")

# 2. 모델 로드
# from_preset을 사용하면 허브에서 가중치와 설정을 자동으로 가져옵니다.
reloaded_hf_model = keras_hub.models.CausalLM.from_preset(hf_model_uri)

# 3. 텍스트 생성 품질을 위한 샘플러 설정
reloaded_hf_model.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))

# 4. 테스트 실행
prompt = "The movie was so"
output = reloaded_hf_model.generate(prompt, max_length=80)

print("\n--- Hugging Face 모델 생성 결과 ---")
print(output)

### 배포된 모델 추론 테스트

Hugging Face Hub에 업로드된 모델을 URI(`hf://...`)를 통해 직접 불러와서 성능을 확인해 봅니다.

In [ ]:
import keras_hub

# 배포된 모델의 Hugging Face URI
model_uri = f"hf://{hf_username}/{model_id}"

print(f"모델 로드 중: {model_uri}")
# 허브에서 모델 로드
test_model = keras_hub.models.CausalLM.from_preset(model_uri)

# 더 자연스러운 텍스트 생성을 위해 샘플러 설정
test_model.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))

# 테스트 프롬프트 입력
prompt = "I thought this movie was"

# 텍스트 생성
output = test_model.generate(prompt, max_length=100)

print("\n--- 생성 결과 ---")
print(output)

### Local vs. Remote Model Performance Comparison

In this section, we compare the generation results of the locally saved model and the model reloaded from Hugging Face Hub to ensure consistency and analyze any differences.

In [ ]:
import time

def run_benchmark(model, prompt, name):
    print(f"\n--- Benchmarking {name} ---")
    start_time = time.time()
    # Ensure consistent sampler settings for fair comparison
    model.compile(sampler=keras_hub.samplers.TopPSampler(p=0.9, temperature=0.7))
    output = model.generate(prompt, max_length=100)
    end_time = time.time()
    print(f"Generation Time: {end_time - start_time:.2f} seconds")
    print(f"Output: {output}")
    return output

# 1. Load Local Model
print("Loading local model...")
local_model = keras_hub.models.CausalLM.from_preset("./my_finetuned_gpt2")

# 2. Load Remote Model (from HF)
print(f"Loading remote model from {hf_model_uri}...")
remote_model = keras_hub.models.CausalLM.from_preset(hf_model_uri)

comparison_prompt = "The plot of this movie is"

# Run tests
local_result = run_benchmark(local_model, comparison_prompt, "Local Model")
remote_result = run_benchmark(remote_model, comparison_prompt, "Hugging Face Model")

# Summary
print("\n--- Comparison Summary ---")
if local_result == remote_result:
    print("✅ Outputs are identical. The upload/download process preserved model integrity.")
else:
    print("⚠️ Outputs differ. This is expected if non-deterministic sampling is used, but the style should remain consistent.")

### 허브에서 가중치 파일 다운로드

배포된 저장소에서 특정 파일(`model.weights.h5`)만 로컬로 다운로드하는 방법입니다.

In [ ]:
from huggingface_hub import hf_hub_download

# 다운로드할 파일 정보 설정
repo_id = f"{hf_username}/{model_id}"
filename = "model.weights.h5"
local_dir = "./downloaded_model"

# 파일 다운로드
downloaded_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    local_dir=local_dir
)

print(f"가중치 파일이 다음 경로로 다운로드되었습니다: {downloaded_path}")

### 다운로드한 가중치 파일 로드하기

특정 경로에 저장된 가중치 파일을 현재 모델 객체에 적용합니다.

In [ ]:
# 1. 모델과 백본을 분리하여 생성합니다.
new_model = keras_hub.models.CausalLM.from_preset("gpt2_base_en")

# 2. 백본(Backbone)에 직접 가중치를 로드합니다.
# .load_weights()의 skip_mismatch=True 옵션을 사용하면 레이어 이름 불일치 문제를 방지할 수 있습니다.
try:
    new_model.backbone.load_weights(downloaded_path)
    print("백본 가중치 로드 완료!")
except Exception as e:
    print(f"가중치 로드 중 오류 발생: {e}")
    print("대안: skip_mismatch=True를 시도합니다.")
    new_model.backbone.load_weights(downloaded_path, skip_mismatch=True)

print("최종 가중치 적용 완료!")

# 3. 로드된 모델로 테스트 수행
print("테스트 생성:", new_model.generate("I love this", max_length=30))